<h2>1. Trenowanie modelu</h2>

<p>Założenia:</p>
<p>Trenowanych będzie 7 modeli dla różnych wartości hiperparametrów:</p>
<ul>
<li>learning rate - współczynnik uczenia</li>
<li>weight decay - parameter regularyzacji L2</li>
<li>batch size - rozmiar batch'a danych</li>
</ul>
<ul>
<li>Każdy model będzie trenowany na zbiorze treningowo-walidującym - 5 razy (każdy fold)</li>
<li>To daje 35 treningów sieci</li>
<li>Hiperparametry najlepszego modelu pod względem średniego AUC, zostaną użyte do końcowego treningu modelu, na cały zbiorze treningowym, bez walidacji, liczba epok treningu modelu końcowego będzie średnią arytmetyczną epok z najlepszej wersji modelu</li>
</ul>
<p>Notebook został przygotowany, aby działał w środowisku Colab</p>
<p>Trening będzie przeprowadzony na jednostkach GPU NVIDIA A100<p>

<h3>1.1. Import bibliotek</h3>

In [ ]:
from __future__ import division, print_function

import os
import tarfile
from glob import glob
from os import makedirs
from shutil import move, rmtree

import pandas as pd
from google.colab import drive

from dxtorchx.training import (
    is_gpu,
    TrainDataset,
    ValidationDataset,
    NeuralNetwork,
    auc_roc_metric,
    evaluate_dataset,
    IMAGES_DIR,
    TEST_SET_FILE,
    fold_train_set_file,
    fold_val_set_file,
    fold_results_dir,
    fold_auc_metrics_file,
    fold_y_true_file,
    fold_y_probability_file
)
from dxtorchx.model_const import densenet121
from dxtorchx.logging import Logger

<h3>1.2. Definicja stałych i wersje modelu</h3>
<p>Model wersja V1</p>
<ul>
<li>learning rate = 0.01</li>
<li>weight decay = 1e-4</li>
<li>batch size = 16</li>
</ul>
<p>Model wersja V2</p>
<ul>
<li>learning rate = 0.003</li>
<li>weight decay = 1e-4</li>
<li>batch size = 16</li>
</ul>
<p>Model wersja V3</p>
<ul>
<li>learning rate = 0.001</li>
<li>weight decay = 1e-4</li>
<li>batch size = 16</li>
</ul>
<p>Model wersja V4</p>
<ul>
<li>learning rate = 0.03</li>
<li>weight decay = 1e-4</li>
<li>batch size = 16</li>
</ul>
<p>Model wersja V5</p>
<ul>
<li>learning rate = najlepszy z poprzednich modeli</li>
<li>weight decay = 5e-4</li>
<li>batch size = 16</li>
</ul>
<p>Model wersja V6</p>
<ul>
<li>learning rate = najlepszy z poprzednich modeli</li>
<li>weight decay = 0</li>
<li>batch size = 16</li>
</ul>
<p>Model wersja V7</p>
<ul>
<li>learning rate = najlepszy z poprzednich modeli</li>
<li>weight decay = najlepszy z poprzednich modeli</li>
<li>batch size = 128</li>
</ul>


In [ ]:
# wersja modelu
MODEL_VERSION = "v1"  # LR=0.01, WD=1e-4, BS=16
# MODEL_VERSION = "v2" # LR=0.003, WD=1e-4, BS=16
# MODEL_VERSION = "v3" # LR=0.001, WD=1e-4, BS=16
# MODEL_VERSION = "v4" # LR=0.03, WD=1e-4, BS=16
# MODEL_VERSION = "v5" # LR=0.01, WD=5e-4, BS=16
# MODEL_VERSION = "v6" # LR=0.01, WD=0, BS=16
# MODEL_VERSION = "v7" # LR=0.08, WD=1e-4, BS=128 | # 0.01 dostosowany do rozmiaru batch'a; learning_rate = learning_rate * (new_batch_size / old_batch_size)

# początkowy krok zmiany wag w kierunku przeciwnym do gradientu
LEARNING_RATE = 0.01
# parametr regularyzacji L2 - weight decay
WEIGHT_DECAY = 1e-4
# rozmiar batcha danych (16 obrazów w batchu)
BATCH_SIZE = 16

<h3>1.3. Montuje Dysk Google, ponieważ tam znajdują się archiwa danych</h3>

In [ ]:
drive.mount("/content/drive")

<h3>1.4. Kopiuje pliki CSV z opisem zbiorów, zbiór treningowy i walidujący dla każdego wyznaczonego fold'a</h3>

In [ ]:
!cp /content/drive/MyDrive/_DATA_/nih_chest_x_ray_dataset/{LABEL_FILE_PREFIX}*.csv .

In [ ]:
!ls

<h3>1.5. Kopiuje archiwa TAR z obrazami (12 plików TAR)</h3>

In [ ]:
!rsync -av /content/drive/MyDrive/_DATA_/nih_chest_x_ray_dataset/archives/ .

<h3>1.6. Sprawdzam czy wszystkie archiwa są skopiowane na lokalny dysk CoLAB</h3>

In [ ]:
!ls

<h3>1.7. Rozpakowuje archiwa ze zdjeciami do katalogów, zgodnymi z nazwami archiwów: <code>images_01</code>, <code>images_02</code> itp.</h3>

In [ ]:
archives = [
    "images_01.tar.gz",
    "images_02.tar.gz",
    "images_03.tar.gz",
    "images_04.tar.gz",
    "images_05.tar.gz",
    "images_06.tar.gz",
    "images_07.tar.gz",
    "images_08.tar.gz",
    "images_09.tar.gz",
    "images_10.tar.gz",
    "images_11.tar.gz",
    "images_12.tar.gz",
]


def extract_tar(archive):
    print(f"Extracting {archive}... ")
    tar = tarfile.open(f"{archive}")
    tar.extractall(archive.split(".")[0])
    tar.close()
    print("Done!")


for archive in archives:
    extract_tar(archive)

In [ ]:
!ls

<h3>1.8. Przenosze wszystkie zdjęcia do jednego katalogu <code>images</code></h3>

In [ ]:
if not os.path.exists(IMAGES_DIR):
    makedirs(IMAGES_DIR)

# Liczba zdjęć 112120
total = 0
for archive in archives:
    dir_name = f"{archive.split('.')[0]}/{IMAGES_DIR}"
    print(dir_name)
    images = glob(f"{dir_name}/*")
    print(len(images))
    total += len(images)
    for image in images:
        file_name = image.split("/")[-1]
        assert file_name.endswith(".png")
        move(image, str(os.path.join(IMAGES_DIR, file_name)))
print(f"Liczba przeniesionych zdjęć: {total}")

<h3>1.9. Potwierdzam, że w katalogu <code>images</code> są wszystkie zdjęcia</h3>

In [ ]:
images = [img.split("/")[-1] for img in glob(f"{IMAGES_DIR}/*")]
print(f"Liczba plików obrazów: {len(images)}")

<h3>1.10. Usuwam puste katalogi</h3>

In [ ]:
for archive in archives:
    dir_name = archive.split(".")[0]
    if os.path.exists(dir_name):
        rmtree(dir_name)
print("Gotowe!")

In [ ]:
!ls

<h3>1.11. Weryfikacja zdjęć ze zbioru CSV vs pliki obrazów na dysku</h3>
<p>Sprawdzam czy zdjęcia wymienione w plikach <code>CSV</code> zbioru testowego, treningowego i walidującego oraz te w katalogu <code>images</code> się zgadzają.</p>

In [ ]:
test_df = pd.read_csv(TEST_SET_FILE)
for fold in range(1, 6):  # 5 foldów
    print(f"FOLD: {fold}")
    train_df = pd.read_csv(fold_train_set_file(fold))
    val_df = pd.read_csv(fold_val_set_file(fold))
    all_df = pd.concat([test_df, train_df, val_df])
    print(f"Liczba zdjęć w zbiorze CSV: {len(all_df)}")
    # Liczba zdjec musi się zgadzać z liczba wymienioną w pliku i musza to byc te same zdjecia
    assert len(all_df) == len(images)
    assert len(set(all_df["Image Index"]) & set(images)) == len(images)
    print("Weryfikacja poprawna!")

    patients = all_df["Patient ID"].unique()
    print(f"Liczba pacjentów: {len(patients)}")
    print("-" * 60)

<h2>2. Uruchamiam trening modelu z wybranymi hiperparametrami</h2>

In [ ]:
if is_gpu():
    Logger.log_number_of_gpu()
for fold in range(1, 6):
    print(f"Fold: {fold}")
    train_ds = TrainDataset(fold=fold)
    val_ds = ValidationDataset(fold=fold)
    model_to_train = densenet121()
    # utworzenie sieci neuronowej z odpowiednimi dla trenowanego modelu hiperparametrami
    neural_network = NeuralNetwork(
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        batch_size=BATCH_SIZE,
        model=model_to_train,
        train_dataset=train_ds,
        validation_dataset=val_ds,
        fold=fold,
        version=MODEL_VERSION,
    )
    # trenowanie
    best_model, best_epoch = neural_network.train()

    # ewaluacja modelu na zbiorze walidującym
    model = densenet121()
    model.load_state_dict(best_model)
    Y_true, Y_probability = evaluate_dataset(
        trained_model=model,
        dataset=val_ds,
        batch_size=BATCH_SIZE,
    )
    # zapisuje orginalne labelki (Ground Truth)
    Y_true.to_csv(
        str(os.path.join(fold_results_dir(fold, MODEL_VERSION), fold_y_true_file(fold, MODEL_VERSION))), index=False
    )
    # zapisuje prawdopodobieństwa (network outputs)
    Y_probability.to_csv(
        str(os.path.join(fold_results_dir(fold, MODEL_VERSION), fold_y_probability_file(fold, MODEL_VERSION))),
        index=False,
    )

    # obliczanie metryki AUC ROC
    auc_roc_df = auc_roc_metric(
        Y_true=Y_true, Y_probability=Y_probability
    )
    # i zapis wyników do pliku CSV
    auc_roc_df.to_csv(
        str(os.path.join(fold_results_dir(fold, MODEL_VERSION), fold_auc_metrics_file(fold, MODEL_VERSION))), index=False
    )

<h3>2.1 Sprawdzam czy wyniku treningu zapisały się na lokalnym dysku Colab</h3>

In [ ]:
!ls {RESULTS_DIR(1, MODEL_VERSION)}/

In [ ]:
!ls {RESULTS_DIR(2, MODEL_VERSION)}/

In [ ]:
!ls {RESULTS_DIR(3, MODEL_VERSION)}/

In [ ]:
!ls {RESULTS_DIR(4, MODEL_VERSION)}/

In [ ]:
!ls {RESULTS_DIR(5, MODEL_VERSION)}/

<h2>2.2 Tworzę katalogi na wyniki na swoim dysku Google oraz kopiuje wyniki ewaluacji modelu na zbiorze walidującym zapisane podczas treningu:</h2>
<ul>
<li>model z epoki w której wartość funkcji kosztu na zbiorze walidującym była najmniejsza</li>
<li>log z treningu</li>
<li>wartości prawdopodobieństw dla wszystkich zdjęć ze zbioru walidującego</li>
<li>metryki AUC dla każdej klasy w zbiorze walidującym dla danego folda</li>
</ul>

In [ ]:
# Tworzę katalogi na dysku Google na wyniki każdego folda
for fold in range(1, 6):
    target_path = f"/content/drive/MyDrive/_DATA_/nih_chest_x_ray_dataset/{fold_results_dir(fold, MODEL_VERSION)}"
    rmtree(target_path, ignore_errors=True)
    os.makedirs(target_path, exist_ok=True)

In [ ]:
!cp {RESULTS_DIR(1, MODEL_VERSION)}/* /content/drive/MyDrive/_DATA_/nih_chest_x_ray_dataset/{RESULTS_DIR(1, MODEL_VERSION)}

In [ ]:
!cp {RESULTS_DIR(2, MODEL_VERSION)}/* /content/drive/MyDrive/_DATA_/nih_chest_x_ray_dataset/{RESULTS_DIR(2, MODEL_VERSION)}

In [ ]:
!cp {RESULTS_DIR(3), MODEL_VERSION}/* /content/drive/MyDrive/_DATA_/nih_chest_x_ray_dataset/{RESULTS_DIR(3, MODEL_VERSION)}

In [ ]:
!cp {RESULTS_DIR(4, MODEL_VERSION)}/* /content/drive/MyDrive/_DATA_/nih_chest_x_ray_dataset/{RESULTS_DIR(4, MODEL_VERSION)}

In [ ]:
!cp {RESULTS_DIR(5, MODEL_VERSION)}/* /content/drive/MyDrive/_DATA_/nih_chest_x_ray_dataset/{RESULTS_DIR(5, MODEL_VERSION)}